In [ ]:
!rm -rf AttentionIsAllINeed
!git clone https://github.com/ivanrj7j/AttentionIsAllINeed

In [ ]:
import os
import sys

os.makedirs("/kaggle/working/AttentionIsAllINeed/runs", exist_ok = True)
os.makedirs("/kaggle/working/AttentionIsAllINeed/checkpoints", exist_ok = True)
sys.path.append('/kaggle/working/AttentionIsAllINeed/src')
os.chdir('/kaggle/working/AttentionIsAllINeed/src')

In [ ]:
import config
config.TRAIN_WORKERS = 4
config.TRAIN_BATCHES = 76000/2
config.TRANSLATE_EVERY = 1000
config.BATCH_SIZE = 128
config.EPOCHS = 2

In [ ]:
print(config.BATCH_SIZE, config.NUM_LAYERS, config.HEAD_COUNT)

In [ ]:
from main import *

In [ ]:
savedModelPath = '/kaggle/input/models/dudefacts/model-e1-5/pytorch/default/1/translator-model.pt'
transformer.load_state_dict(torch.load(savedModelPath, weights_only=True))
transformer.to('cpu')
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    transformer = nn.DataParallel(transformer)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
transformer.to(device)

In [ ]:
from IPython.display import FileLink, display
if __name__ == '__main__':
    globalSteps = 0

    # Load one fixed batch for quick "live" checks
    val_iter = iter(validationDataLoader)
    fixed_val_src, fixed_val_tgt = next(val_iter)
    fixed_val_src = fixed_val_src.to(config.TARGET_DEVICE)
    fixed_val_tgt = fixed_val_tgt.to(config.TARGET_DEVICE)
    
    for epoch in range(config.EPOCHS):
        transformer.train()
        
        epochStart = time.time()
        bar = tqdm(enumerate(trainingDataLoader), total=config.TRAIN_BATCHES, desc=f"Epoch {epoch+1}/{config.EPOCHS}")
        for i, (srcBatch, tgtBatch) in bar:
            if i >= config.TRAIN_BATCHES:
                break
            optimizer.zero_grad()
            batchStart = time.time()
            srcBatch, tgtBatch = srcBatch.to(config.TARGET_DEVICE), tgtBatch.to(config.TARGET_DEVICE)
            with torch.amp.autocast_mode.autocast(device_type='cuda', dtype=torch.float16):
                output = transformer(srcBatch, tgtBatch[:, :-1])
                loss = criterion(output.contiguous().view(-1, config.TGT_VOCAB_SIZE), tgtBatch[:, 1:].contiguous().view(-1))
            
            if torch.isnan(loss):
                # Convert tokens back to strings to see what caused it
                src_text = srcTokenizer.decode(srcBatch.detach().cpu().numpy())
                tgt_text = tgtTokenizer.decode(tgtBatch.detach().cpu().numpy())
                print(f"!!! NaN detected at batch {i} !!!", src_text, tgt_text)
                
                with open("nan_debug_log.txt", "a") as f:
                    f.write(f"Batch {i} poisoned:\n")
                    f.write(f"SRC: {src_text}\n")
                    f.write(f"TGT: {tgt_text}\n\n")
                
                # Reset gradients and skip this batch
                optimizer.zero_grad()
                continue

            scaler.scale(loss).backward()
            # Unscales the gradients of optimizer's assigned params in-place
            scaler.unscale_(optimizer)
            # loss.backward()

            # Clip the gradients (Max norm of 1.0 is standard)
            torch.nn.utils.clip_grad_norm_(transformer.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            # optimizer.step()
            scheduler.step()
            t = time.time() - batchStart
            currentLR = optimizer.param_groups[0]['lr']
            bar.set_postfix({
                "loss": loss.item(),
                "lr": currentLR
            })
            if globalSteps % config.LOG_EVERY == 0 and globalSteps > 0:
                summaryWriter.add_scalar('Charts/BatchLR', currentLR, globalSteps)
                summaryWriter.add_scalar('Charts/BatchLoss', loss.item(), globalSteps)
                summaryWriter.add_scalar('Charts/BatchTime', t, globalSteps)

            if globalSteps % 4000 == 0 and globalSteps > 0:
                savedPath = os.path.join(config.CHECKPOINT_PATH, f'transformer--{epoch}-{globalSteps}.pt')

                torch.save(transformer.state_dict(), savedPath)
                display(FileLink(savedPath))
                print("Zip this one", savedPath)
                # saving the model every 5000 epochs 

            if globalSteps % config.TRANSLATE_EVERY == 0 and globalSteps > 0:
                transformer.eval()
                with torch.no_grad():
                    # Pick a random sample from our pre-loaded batch
                    idx = random.randint(0, fixed_val_src.size(0) - 1)
                    s_ten, t_ten = fixed_val_src[idx:idx+1], fixed_val_tgt[idx:idx+1]

                    with torch.amp.autocast_mode.autocast(device_type='cuda', dtype=torch.float16):
                        output = transformer(s_ten, t_ten[:, :-1])
                    
                    # Decode and clean strings
                    src_str = srcTokenizer.decode(s_ten[0].tolist(), skip_special_tokens=True)
                    tgt_str = tgtTokenizer.decode(t_ten[0].tolist(), skip_special_tokens=True)
                    pred_str = tgtTokenizer.decode(output.argmax(dim=-1)[0].tolist(), skip_special_tokens=True)

                    # Single-line output to keep the terminal clean
                    print(f"Step {globalSteps} | SRC: {src_str[:40]}... | PRD: {pred_str[:40]}...")
                    
                    summaryWriter.add_text('Samples/Validation', f"**SRC:** {src_str}  \n**TGT:** {tgt_str}  \n**PRD:** {pred_str}", globalSteps)

                transformer.train()
            globalSteps += 1

        epochTime = time.time() - epochStart
        if epoch % config.SAVE_MODEL_EVERY == 0 and config.EPOCHS > 1:
            torch.save(transformer.state_dict(), os.path.join(config.CHECKPOINT_PATH, f'transformer-{epoch+1}.pt'))

        summaryWriter.add_scalar('Charts/loss', loss.item(), epoch)
        summaryWriter.add_scalar('Charts/time', epochTime)
        summaryWriter.add_scalar('Charts/lr', currentLR)

        valLoss = run_validation(
                    transformer, 
                    validationDataLoader, 
                    criterion, 
                    config.TARGET_DEVICE, 
                    config.TGT_VOCAB_SIZE,
                    srcTokenizer,    
                    tgtTokenizer,    
                    summaryWriter,   
                    totalSteps,      
                    max_batches=20
                )
        summaryWriter.add_scalar('Charts/valLoss', valLoss, epoch)

    savedPath = os.path.join(config.CHECKPOINT_PATH, f'transformer-final.pt')

    torch.save(transformer.state_dict(), savedPath)
    display(FileLink(savedPath))

In [ ]:
os.listdir(config.CHECKPOINT_PATH)